# 🥈 NYC Taxi Transformation - Silver Layer
> **Goal:** Clean raw data, enforce schema, and perform an incremental **Upsert (Merge)** into the Silver table. 🚕✨

### 🛠️ Engineering Steps:
1. **Load** new data from the Bronze layer.
2. **Clean** column names and filter out invalid records (e.g., zero distance or negative fares).
3. **Deduplicate** within the incoming batch.
4. **Merge (Upsert)** into the Silver Delta table using `VendorID` and `tpep_pickup_datetime` as keys. 🔄

In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Load Data from Bronze
df_bronze = spark.read.table("bronze_nyc_taxi")

# 2. Data Cleaning & Transformation
# - Filter for trips with valid distance and fare
# - Ensure column names are clean (standardize case)
df_silver_updates = df_bronze.filter("trip_distance > 0 AND total_amount > 0") \
                             .dropDuplicates(["VendorID", "tpep_pickup_datetime"])

# 3. Incremental Refresh Logic (The Delta Merge)
target_table_name = "silver_nyc_taxi"

# Check if the table exists to decide between Create or Merge
if not spark.catalog.tableExists(target_table_name):
    # First time run: Create the table
    df_silver_updates.write.format("delta").mode("overwrite").saveAsTable(target_table_name)
    print(f"✅ Created initial Silver table: {target_table_name}")
else:
    # Subsequent runs: Perform the Merge (Upsert)
    print(f"🔄 Performing Incremental Merge into {target_table_name}...")
    
    delta_table = DeltaTable.forName(spark, target_table_name)
    
    delta_table.alias("target").merge(
        df_silver_updates.alias("updates"),
        "target.VendorID = updates.VendorID AND target.tpep_pickup_datetime = updates.tpep_pickup_datetime"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print("✅ Incremental Merge complete!")

StatementMeta(, ec360242-4aa0-4084-b7f4-efe41c3a915c, 4, Finished, Available, Finished, False)

✅ Created initial Silver table: silver_nyc_taxi
